# TAREA SEMANA 5: RECONOCIMIENTO DE VOZ

> **Basado en:** `ReconocimientoDeVoz/speakerrecognition.ipynb`  
> **Hablantes:** Voz propia · Julio Cortázar · Jorge Luis Borges · Silencio

## Introducción

El **reconocimiento de hablantes** y el **reconocimiento de voz** son dos campos distintos pero relacionados dentro del dominio más amplio del procesamiento y análisis de señales de audio. A continuación, se presentan las principales diferencias entre ellos:

1. **Objetivo:**
   - **Reconocimiento de hablantes:** El objetivo principal es identificar o verificar la identidad de un hablante en función de sus características vocales únicas, conocidas como "huellas de voz" o "firmas biométricas". Es una forma de autenticación biométrica.
   - **Reconocimiento de voz:** El objetivo principal es convertir el lenguaje hablado en texto u otros tipos de comandos. Los sistemas de reconocimiento de voz analizan señales de audio para comprender y transcribir las palabras pronunciadas.

2. **Enfoque:**
   - **Reconocimiento de hablantes:** Se centra en las características únicas de la voz de una persona, como el tono, timbre, acento y patrones de habla, para establecer su identidad.
   - **Reconocimiento de voz:** Se centra en comprender e interpretar el contenido lingüístico de las palabras habladas, sin importar quién las diga.

3. **Aplicaciones:**
   - **Reconocimiento de hablantes:** Se utiliza comúnmente en sistemas de seguridad, control de acceso y aplicaciones de autenticación donde se debe verificar la identidad del hablante.
   - **Reconocimiento de voz:** Se aplica en asistentes activados por voz, servicios de transcripción, comandos de voz en dispositivos inteligentes y sistemas de respuesta de voz interactiva (IVR).

4. **Desafíos:**
   - **Reconocimiento de hablantes:** Afronta desafíos como las variaciones de la voz debido a la salud, el estado emocional o las condiciones ambientales. También debe tener en cuenta posibles intentos de suplantación de voz.
   - **Reconocimiento de voz:** Sus desafíos incluyen manejar variaciones de acento, ruido de fondo y la comprensión del lenguaje dependiente del contexto.

5. **Técnicas:**
   - **Reconocimiento de hablantes:** Utiliza técnicas como verificación de hablante (confirmar identidad) e identificación de hablante (nombrar al hablante) basadas en extracción de características y emparejamiento de patrones.
   - **Reconocimiento de voz:** Utiliza técnicas como Modelos Ocultos de Markov (HMM), redes neuronales profundas (DNN) y redes neuronales recurrentes (RNN) para el modelado acústico y lingüístico.

6. **Salida:**
   - **Reconocimiento de hablantes:** Entrega la identidad o el resultado de verificación del hablante.
   - **Reconocimiento de voz:** Entrega el texto transcrito o los comandos hablados reconocidos.

## **Planteamiento del problema:**

En este proyecto contamos con archivos `.wav` de **1 segundo** para **4 clases** de audio:

| Clase | Hablante | Fuente | Clips |
|---|---|---|---|
| `yo` | Voz propia | Grabadora de Windows (.m4a → .wav) | 195 |
| `persona2` | Julio Cortázar | YouTube — "Me caigo y me levanto" | 228 |
| `persona3` | Jorge Luis Borges | YouTube — entrevista | 350 |
| `silencio` | Silencio ambiental | Grabadora de Windows | 116 |

El objetivo es entrenar un modelo **RNN (Red Neuronal Recurrente)** para predecir si el modelo logra capturar correctamente al hablante.

Las **Redes Neuronales Recurrentes (RNN)** son especialmente adecuadas para ciertos aspectos del **reconocimiento de hablantes** debido a su capacidad para:
- Modelar dependencias secuenciales.
- Capturar patrones temporales en los datos de audio.

## IMPORTANDO LIBRERÍAS

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
import seaborn as sns
import tensorflow as tf

from IPython.display import display, Audio
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))

Los audios fueron recolectados manualmente y procesados con **librosa** y **soundfile** para crear los clips de 1 segundo.

- **librosa**: es un paquete de Python para el análisis de música y audio. Proporciona herramientas para analizar y visualizar datos de audio, incluyendo funciones para extracción de características, representación en series de tiempo y visualización de señales de audio. Es ampliamente utilizado en el campo de la recuperación de información musical y el procesamiento de señales de audio.

- **soundfile**: es una librería de Python para leer y escribir archivos de sonido. Ofrece una interfaz sencilla para trabajar con archivos de audio y soporta varios formatos como WAV, FLAC y OGG. A menudo se usa junto con **librosa** al trabajar con datos de audio, ya que facilita la carga y el guardado eficiente de archivos de audio.

## Verificar clips por clase

In [ ]:
# Carpeta raíz del dataset
parent_dir = "audio_data"

# Nombres de las carpetas (usados para leer los archivos)
speaker_folders = ["yo", "persona2", "persona3", "silencio"]

# Nombres reales para mostrar en gráficas y tablas
class_names = ["Yo", "Julio Cortazar", "Jorge Luis Borges", "Silencio"]

# Archivo de audio fuente por clase (para visualización)
source_files = {
    "yo":       "audio_data/yo/voz_yo.wav",
    "persona2": "audio_data/persona2/Me caigo y me levanto_Julio Cortazar.wav",
    "persona3": "audio_data/persona3/Borges explica el origen y recita su mejor poema - Jorge Luis Borges.wav",
    "silencio": "audio_data/silencio/silencio.wav"
}

for carpeta, nombre in zip(speaker_folders, class_names):
    clips = [f for f in os.listdir(f"{parent_dir}/{carpeta}")
             if f.endswith(".wav") and f.replace(".wav", "").isdigit()]
    print(f"{nombre}: {len(clips)} clips de 1 segundo")

**IPython** es una **consola interactiva de línea de comandos para Python**.  
Proporciona un entorno interactivo mejorado para la programación en Python y es particularmente popular entre **científicos de datos, investigadores e ingenieros** que trabajan en **computación científica, análisis de datos y aprendizaje automático**.

In [ ]:
# Reproducir un clip de muestra de cada clase
for carpeta, nombre in zip(speaker_folders, class_names):
    print(f"Haz clic en el botón para escuchar: {nombre}")
    display(Audio(filename=f"{parent_dir}/{carpeta}/0.wav"))

## VISUALIZACIONES DE DATOS DE AUDIO

Para cada clase se muestran tres representaciones del audio. Cada descripción va seguida del código que la genera.

---

### 1. Forma de onda (*Waveform*)
- **Qué es:** Representación en el **dominio del tiempo** de la señal de audio.
- **Ejes:** Eje X: tiempo. Eje Y: amplitud (volumen o intensidad de la señal).
- **Qué muestra:** Cómo cambia la amplitud del sonido a lo largo del tiempo.
- **Interpretación:** Los **picos y valles** representan variaciones en la presión del aire que percibimos como sonido.

In [ ]:
for carpeta, nombre in zip(speaker_folders, class_names):
    y, sr = librosa.load(source_files[carpeta], sr=None)
    plt.figure(figsize=(15, 3))
    librosa.display.waveshow(y, sr=sr)
    plt.title(f'Forma de onda - {nombre}')
    plt.tight_layout()
    plt.show()

### 2. Espectrograma (*Spectrogram*)
- **Qué es:** Representación en el **dominio de la frecuencia**.
- **Ejes:** Eje X: tiempo. Eje Y: frecuencia. Color: energía/magnitud de cada frecuencia.
- **Qué muestra:** La distribución del contenido frecuencial del audio a lo largo del tiempo.
- **Interpretación:** Las zonas más intensas indican frecuencias con mayor energía en ese instante.

In [ ]:
for carpeta, nombre in zip(speaker_folders, class_names):
    y, sr = librosa.load(source_files[carpeta], sr=None)
    S = librosa.stft(y)
    D = librosa.amplitude_to_db(np.abs(S), ref=np.max)
    plt.figure(figsize=(15, 3))
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='log')
    plt.colorbar(format='%+2.0f dB')
    plt.title(f'Espectrograma - {nombre}')
    plt.tight_layout()
    plt.show()

### 3. MFCCs (*Mel-Frequency Cepstral Coefficients*)
Los MFCCs son una representación del audio en el **dominio de la frecuencia** percibida por el oído humano. Son la base de entrada para el modelo LSTM.

- **Ejes del gráfico:** Eje X: tiempo. Eje Y: coeficiente MFCC (banda de frecuencia).
- **Extracción de características:** Cada fila representa un coeficiente que recoge propiedades del tracto vocal en ese instante.
- **Análisis de patrones:** La variación de los coeficientes en el tiempo permite identificar patrones de voz propios de cada hablante.

In [ ]:
for carpeta, nombre in zip(speaker_folders, class_names):
    y, sr = librosa.load(source_files[carpeta], sr=None)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    plt.figure(figsize=(15, 3))
    librosa.display.specshow(mfccs, x_axis='time')
    plt.colorbar()
    plt.title(f'MFCCs - {nombre}')
    plt.tight_layout()
    plt.show()

## Comparación de características de voz

| Característica | Descripción | Yo | Julio Cortázar | Jorge Luis Borges | Silencio |
|---|---|---|---|---|---|
| **Forma de onda (Waveform)** | Representa la amplitud en el tiempo. Permite ver intensidad, pausas y fluidez del habla. | Amplitud variable con pausas naturales del habla cotidiana. | Onda expresiva con cambios bruscos de intensidad → habla dramática y enfática. | Onda más uniforme y pausada → habla reflexiva y controlada. | Amplitud casi cero → solo ruido de fondo constante. |
| **Espectrograma** | Muestra la distribución de frecuencias a lo largo del tiempo, con colores según intensidad. | Energía concentrada en frecuencias medias → acento local característico. | Rico en bajas y medias con silencios marcados → estilo teatral. | Patrones homogéneos y lentos → entonación estable y formal. | Sin estructura vocal, energía distribuida uniformemente. |
| **MFCCs** | Representación de cómo el oído humano percibe el sonido. Base para ML en reconocimiento de voz. | Variaciones propias del acento regional. | Alta varianza en coeficientes → habla muy expresiva. | Coeficientes más estables → habla pausada y monótona. | Coeficientes casi planos → ausencia de patrón vocal. |
| **Estilo de habla** | Interpretación general basada en las visualizaciones. | Habla coloquial y natural. | Dramático, con énfasis y pausas teatrales. | Reflexivo, lento y académico. | Ausencia de voz. |

---

## Conclusión
Cada clase presenta una firma MFCC suficientemente distinta para que el modelo LSTM aprenda a diferenciarlas.

## Extracción de características

La **extracción de características** es un paso fundamental en la preparación de datos para tareas de *Machine Learning* e Inteligencia Artificial. Su importancia radica en varios aspectos clave:

1. **Reducción de dimensionalidad:**
   Los datos en bruto (como audio, imágenes o texto) suelen ser muy complejos y de alta dimensión.
   Al extraer características relevantes, reducimos la cantidad de información a procesar, haciéndola más manejable y eficiente computacionalmente.

2. **Captura de información relevante:**
   No toda la información de los datos originales es útil.
   La extracción de características nos ayuda a identificar y conservar solo los elementos más importantes para la tarea específica.

3. **Reducción de ruido:**
   Al enfocarnos en atributos concretos, podemos filtrar detalles irrelevantes o ruidos presentes en los datos en bruto,
   lo que mejora la robustez y precisión de los modelos.

4. **Mejor desempeño de los modelos:**
   Un conjunto de características bien definidas permite que los modelos generalicen mejor y realicen predicciones más precisas.

5. **Facilita el aprendizaje:**
   Las características extraídas resaltan patrones, relaciones y estructuras en los datos
   que hacen que el modelo aprenda más rápido y de forma más efectiva.

6. **Interpretabilidad humana:**
   En muchos casos, las características extraídas son más fáciles de interpretar por humanos que los datos originales,
   lo que ayuda a comprender cómo funciona un modelo y a confiar en sus predicciones.

7. **Adaptación al dominio específico:**
   Según el tipo de problema, se pueden extraer diferentes conjuntos de características
   (por ejemplo, **MFCCs en audio**, **bordes en imágenes**, **tokens en texto**).

8. **Manejo de datos multimodales:**
   Cuando los datos provienen de distintas fuentes (audio, texto, imágenes),
   la extracción de características permite integrarlos en una representación común, facilitando el análisis conjunto.

In [ ]:
def extract_features(parent_dir, speaker_folders):
    features = []
    labels = []

    for i, carpeta in enumerate(speaker_folders):
        ruta_carpeta = os.path.join(parent_dir, carpeta)

        # Solo clips numerados (excluye archivos fuente originales)
        archivos = sorted(
            [f for f in os.listdir(ruta_carpeta)
             if f.endswith(".wav") and f.replace(".wav", "").isdigit()],
            key=lambda x: int(x.replace(".wav", ""))
        )
        print(f"Procesando {class_names[i]}: {len(archivos)} clips...")

        for archivo in archivos:
            ruta = os.path.join(ruta_carpeta, archivo)
            try:
                audio, sr = librosa.load(ruta, sr=None, duration=1)

                # Padding si el clip es ligeramente más corto de 1 segundo
                if len(audio) < sr:
                    audio = np.pad(audio, (0, sr - len(audio)))

                mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
                mfccs = StandardScaler().fit_transform(mfccs)
                features.append(mfccs.T)
                labels.append(i)
            except Exception:
                pass

    return np.array(features), np.array(labels)


X, y = extract_features(parent_dir, speaker_folders)

## Interpretación de los Coeficientes MFCC

Los números en el arreglo representan las características **MFCC (Mel-Frequency Cepstral Coefficients)** extraídas para cada **cuadro de audio**.
- Cada **fila** del arreglo corresponde a un cuadro (frame) de audio.
- Cada **columna** corresponde a un coeficiente MFCC específico.

El resultado será un arreglo de números que representan los coeficientes MFCC para el **primer cuadro del primer archivo de audio**.

---

## ¿Qué capturan los MFCC?
Los MFCC contienen información relacionada con la **forma del tracto vocal** durante la producción de sonidos del habla.

- Los **primeros coeficientes** capturan información global (energía, pendiente espectral).
- Los **coeficientes de orden superior** capturan detalles más finos de la estructura espectral.

---

## Interpretación de algunos coeficientes MFCC

| **Coeficiente** | **Descripción** |
|---|---|
| **MFCC 0** | Representa la **energía total** de la señal. |
| **MFCC 1** | Representa la **pendiente espectral global**, relacionado con la **percepción del tono**. |
| **MFCC 2** | Captura la forma del tracto vocal, asociado a **formantes** en el habla. |
| **MFCC 3** | Refleja cambios en la **envolvente espectral**, puede estar relacionado con la **nasalidad**. |
| **MFCC 4 en adelante** | Capturan características espectrales más detalladas, como **estructuras finas del espectro**. |

In [ ]:
# Imprime las primeras características extraídas
for feature in X[:1]:
    print(feature)

## CONFIGURACIÓN DEL MODELO - DIVISIÓN TRAIN/TEST Y EVALUACIÓN

Dividiendo los datos en un 70% para entrenamiento y el restante se dividirá en partes iguales para los conjuntos de validación y prueba.

In [ ]:
label_encoder = LabelEncoder()
y_enc = label_encoder.fit_transform(y)
label_encoder.classes_ = np.array(class_names)

X_train, X_temp, y_train, y_temp = train_test_split(X, y_enc, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("Forma de los datos de entrenamiento:", X_train.shape)
print("Forma de los datos de validacion:   ", X_val.shape)

## TRAINING

## Definiendo el modelo RNN y funciones de activación

Cuando definimos un modelo de **Red Neuronal Recurrente (RNN)**, se utilizan funciones de activación como **ReLU** y **Softmax** en diferentes capas de la red.

### ReLU (Rectified Linear Unit)
- Es una función de activación comúnmente usada en las **capas ocultas** de las redes neuronales.
- Matemáticamente se define como: **f(x) = max(0, x)**
- Introduce **no linealidad** al modelo, permitiendo aprender patrones complejos.

---

### Softmax
- Se utiliza en la **capa de salida** de redes neuronales para problemas de **clasificación multiclase**.
- Convierte las salidas (logits) en una **distribución de probabilidades** sobre todas las clases.
- La clase con mayor probabilidad será la predicción final.

En la arquitectura del modelo, la **última capa usa Softmax** para obtener las probabilidades sobre los 4 hablantes.
La capa anterior usa **ReLU** para aportar no linealidad en la capa oculta.

---

## Funciones de pérdida (Loss Functions)

- **Error Cuadrático Medio (MSE):** Usado en problemas de **regresión**.
- **Binary Crossentropy:** Usado en **clasificación binaria** (0 o 1).
- **Categorical Crossentropy:** Usado en **clasificación multiclase** con variables **one-hot encoded**.
- **Sparse Categorical Crossentropy:** Similar a categorical crossentropy, pero se usa cuando las clases están en formato **índices enteros** (no en one-hot). Evita tener que transformar las etiquetas manualmente.

---

## Elección en nuestro modelo

- El problema es de **clasificación multiclase** (4 clases: Yo, Cortázar, Borges, Silencio).
- Por eso se usa **sparse_categorical_crossentropy** como función de pérdida.
- El optimizador elegido es **Adam**, ya que adapta las tasas de aprendizaje de forma automática y es muy eficiente.

### Capas de una Red Neuronal

En el contexto de configurar capas de una red neuronal, los números **128** y **64** se refieren al número de **neuronas o unidades** en esas capas específicas.

---

#### `tf.keras.layers.LSTM(128, input_shape=(X_train.shape[1], X_train.shape[2]))`

Esta es una capa **LSTM (Long Short-Term Memory)** con **128 unidades**.
Las capas LSTM son un tipo de red neuronal recurrente (**RNN**) especialmente efectivas para datos secuenciales como series temporales o texto.

---

#### `tf.keras.layers.Dense(64, activation='relu')`

Esta es una capa **densa (totalmente conectada)** con **64 unidades** y una función de activación **ReLU**.
Las capas densas conectan cada neurona de una capa con todas las neuronas de la siguiente.

---

#### `tf.keras.layers.Dense(len(speaker_folders), activation='softmax')`

Esta es la **capa de salida**, con tantas unidades como clases haya (`len(speaker_folders)` = 4).
La función de activación **softmax** convierte los valores en **probabilidades** para cada clase.

---

### Ajuste de Hiperparámetros

Las elecciones de **128** y **64** son algo arbitrarias y pueden ajustarse según las características específicas de los datos.
El número de unidades en una capa es un **hiperparámetro** que puede experimentarse durante el ajuste del modelo.
Problemas más complejos pueden requerir más unidades, pero esto también aumenta el riesgo de **sobreajuste** si no se controla adecuadamente.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.LSTM(128, input_shape=(X_train.shape[1], X_train.shape[2])),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(len(speaker_folders), activation='softmax')
])

model.summary()

### Compilación del modelo

La **compilación** configura el proceso de aprendizaje antes de entrenar:
- **Optimizador `adam`:** Ajusta los pesos de forma adaptativa en función del gradiente de cada parámetro.
- **Función de pérdida `sparse_categorical_crossentropy`:** Apropiada para clasificación multiclase con etiquetas enteras (no one-hot).
- **Métrica `accuracy`:** Proporción de predicciones correctas sobre el total de muestras.

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

### Detención Temprana y Entrenamiento

Se usa **EarlyStopping** para evitar sobreajuste: si `val_loss` no mejora durante 3 épocas consecutivas, el entrenamiento se detiene y se restauran automáticamente los mejores pesos (`restore_best_weights=True`).

El modelo se entrena con un máximo de **30 épocas** y lotes de **32 muestras**, monitoreando el progreso con el conjunto de validación.

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=[early_stopping]
)

if early_stopping.stopped_epoch > 0:
    print("Entrenamiento detenido tempranamente en la epoca", early_stopping.stopped_epoch + 1)
else:
    print("Entrenamiento completado sin detencion temprana")

### Curva de pérdida

Muestra cómo disminuye la **pérdida** en entrenamiento y validación a lo largo de las épocas. Si la pérdida de validación sube mientras la de entrenamiento baja, es señal de sobreajuste.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Perdida de entrenamiento')
plt.plot(history.history['val_loss'], label='Perdida de validacion')
plt.xlabel('Epocas')
plt.ylabel('Perdida')
plt.title('Curva de Perdida')
plt.legend()
plt.tight_layout()
plt.show()

### Curva de precisión

Muestra la **precisión** del modelo en entrenamiento y validación por época. Curvas cercanas y crecientes indican buen aprendizaje; una brecha amplia entre ellas puede señalar sobreajuste.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history['accuracy'], label='Precision de entrenamiento')
plt.plot(history.history['val_accuracy'], label='Precision de validacion')
plt.xlabel('Epocas')
plt.ylabel('Precision')
plt.title('Curva de Precision')
plt.legend()
plt.tight_layout()
plt.show()

## EVALUATION

### Predicciones del modelo

El modelo recibe el conjunto de prueba y devuelve un vector de probabilidades para cada clase. Se selecciona la clase con mayor probabilidad como predicción final (`argmax`). Luego se decodifican los índices numéricos de vuelta a nombres de clase para facilitar la interpretación.

In [ ]:
y_pred_probabilities = model.predict(X_test)
y_pred = np.argmax(y_pred_probabilities, axis=1)

y_test_decoded = label_encoder.inverse_transform(y_test)
y_pred_decoded = label_encoder.inverse_transform(y_pred)

### Métricas de evaluación

- **Accuracy:** Porcentaje de muestras clasificadas correctamente sobre el total del conjunto de prueba.
- **F1-Score ponderado:** Promedio de precisión y recall por clase, ponderado por la cantidad de muestras de cada clase. Es más informativo que el accuracy cuando hay desbalance entre clases.

In [ ]:
accuracy = accuracy_score(y_test_decoded, y_pred_decoded)
print(f"Precision en la evaluacion del conjunto de prueba: {accuracy:.4f} ({accuracy*100:.2f}%)")

f1 = f1_score(y_test_decoded, y_pred_decoded, labels=class_names, average='weighted')
print(f"Puntaje F1 ponderado: {f1:.4f}")

### Matriz de confusión

Cada **fila** representa la clase real y cada **columna** la clase predicha. Los valores en la diagonal son aciertos; los valores fuera de la diagonal son errores. Permite identificar qué clases confunde el modelo entre sí.

In [ ]:
conf_matrix = confusion_matrix(y_test_decoded, y_pred_decoded, labels=class_names)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xticks(rotation=45, ha="right")
plt.title("Matriz de Confusion")
plt.xlabel("Etiqueta Predicha")
plt.ylabel("Etiqueta Verdadera")
plt.tight_layout()
plt.show()

### Nota sobre el uso de la clase Silencio

No utilizamos ruido de fondo explícito como en algunos datasets profesionales, pero la clase **Silencio** cumple un rol equivalente: ayuda al modelo a distinguir entre presencia y ausencia de voz. A continuación se explica por qué esto es importante.

### Importancia de Incluir Ruido de Fondo en los Datos de Entrenamiento

Incluir ruido de fondo en los datos de entrenamiento puede ser importante por varias razones:

---

#### 1. Robustez ante Condiciones del Mundo Real
En escenarios reales, los entornos rara vez son silenciosos. Incluir ruido de fondo ayuda al modelo a volverse más robusto frente a diversas condiciones ambientales, asegurando un buen rendimiento incluso en presencia de ruido, tal como ocurriría en su uso real.

---

#### 2. Generalización a Entornos Diversos
Entrenar con datos que contienen ruido permite que el modelo generalice mejor a diferentes ambientes acústicos. Si el modelo solo aprende con datos limpios, podría tener dificultades para adaptarse a nuevos entornos con distintos niveles y tipos de ruido.

---

#### 3. Evitar el Sobreajuste a Datos Limpios
Los modelos entrenados únicamente con datos limpios pueden volverse demasiado especializados en condiciones ideales y no funcionar bien en situaciones reales con ruido. Incluir ruido de fondo ayuda a prevenir este sobreajuste.

---

#### 4. Mejorar la Privacidad y Seguridad
En aplicaciones como autenticación por voz o reconocimiento de hablantes, es crucial que el modelo sea robusto ante condiciones ruidosas. Esto garantiza que el modelo no sea fácilmente engañado por intentos de imitar una voz limpia.

---

#### 5. Mayor Realismo en la Generación de Datos Sintéticos
Al crear conjuntos de datos sintéticos para entrenamiento, incorporar ruido de fondo hace que los datos sean más realistas. Los datos sintéticos con patrones de ruido realistas pueden ser valiosos, especialmente cuando es difícil obtener grandes cantidades de datos reales.

---

#### 6. Cumplir con Requisitos de la Aplicación
Dependiendo de la aplicación, el modelo puede necesitar operar en entornos con distintos niveles de ruido. Entrenar con datos que incluyan diversas condiciones de ruido asegura que el modelo cumpla con los requisitos específicos de la aplicación.

---

#### 7. Adaptarse a la Variabilidad del Comportamiento del Usuario
Los usuarios pueden interactuar con dispositivos en distintos entornos, y su comportamiento puede introducir ruido de fondo. Entrenar con datos diversos ayuda al modelo a adaptarse a esta variabilidad en el comportamiento del usuario.

## Conclusiones

- Se entrenó exitosamente un modelo **LSTM** para clasificar 4 clases de audio: voz propia, Julio Cortázar, Jorge Luis Borges y silencio.
- El dataset fue construido íntegramente de forma manual con grabaciones propias y audios descargados de YouTube, a diferencia del notebook base que usaba un dataset profesional de 1500 clips por hablante.
- Los **MFCCs** capturaron efectivamente las diferencias vocales: el acento local de la voz propia, el estilo dramático de Cortázar, la cadencia pausada de Borges, y la ausencia de estructura vocal en el silencio.
- La clase **Silencio** tiene menor cantidad de datos (116 clips), lo cual puede reflejarse en menor precisión para esa clase en la matriz de confusión.
- El uso de **EarlyStopping** con `patience=3` evitó el sobreajuste y redujo el tiempo de entrenamiento preservando los mejores pesos.

## Breve Informe: Proceso y Hallazgos

### Proceso

**1. Recolección de datos**  
Se construyó un dataset propio de 889 clips de 1 segundo distribuidos en 4 clases. La voz propia y el silencio fueron grabados con la Grabadora de Windows y convertidos a WAV mono 16 kHz con `ffmpeg`. Los audios de Julio Cortázar ("Me caigo y me levanto") y Jorge Luis Borges (entrevista) fueron descargados de YouTube y procesados con el mismo pipeline.

**2. Preprocesamiento**  
Cada clip fue cortado exactamente a 1 segundo usando `librosa` + `soundfile`. Se aplicó padding con ceros a clips ligeramente más cortos. Para cada clip se extrajeron 13 coeficientes MFCC normalizados con `StandardScaler`, resultando en tensores de forma `(32, 13)` por muestra.

**3. División y entrenamiento**  
El dataset se dividió en 70% entrenamiento, 15% validación y 15% prueba. Se entrenó una red LSTM de arquitectura `LSTM(128) → Dense(64, ReLU) → Dense(4, Softmax)` con el optimizador Adam, `sparse_categorical_crossentropy` como función de pérdida y `EarlyStopping(patience=3)` para evitar sobreajuste.

---

### Hallazgos

**Rendimiento del modelo**  
El modelo logró aprender a distinguir las 4 clases a partir de los MFCCs. Las clases con más datos (Borges: 350, Cortázar: 228) tuvieron mejor representación en el entrenamiento que la clase Silencio (116 clips), lo que se refleja en la matriz de confusión.

**Diferencias vocales capturadas**  
- **Yo:** Los MFCCs mostraron variaciones propias del acento regional y el habla coloquial.
- **Julio Cortázar:** Alta varianza en los coeficientes, consistente con su estilo dramático y expresivo.
- **Jorge Luis Borges:** Coeficientes más estables y uniformes, reflejo de su cadencia pausada y formal.
- **Silencio:** Coeficientes casi planos sin patrón vocal reconocible.

**Limitaciones identificadas**  
- El dataset es significativamente más pequeño que el del notebook base (889 vs 7500 muestras), lo que limita la capacidad de generalización.
- El desbalance entre clases (Borges triplica al Silencio) puede sesgar las predicciones.
- La calidad del audio varía: la voz propia fue grabada en condiciones caseras, mientras que los audios de YouTube tienen compresión de audio.

**Mejoras posibles**  
- Descargar más minutos de audio de Cortázar y Borges para igualar las clases.
- Agregar más grabaciones de silencio y ruido ambiental variado.
- Experimentar con arquitecturas GRU o capas Dropout para mejorar la generalización.